In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import sqlite3
import pandas as pd
import re

# データベース設定（変更なし）
class RealEstateDB:
    def __init__(self, db_name="suumo_data.db"):
        self.conn = sqlite3.connect(db_name)
        self.cursor = self.conn.cursor()
        self.create_table()

    def create_table(self):
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS properties (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT,
                address TEXT,
                price REAL,
                age INTEGER,
                station TEXT
            )
        ''')
        self.conn.commit()

    def insert_data(self, data_list):
        self.cursor.executemany('''
            INSERT INTO properties (name, address, price, age, station)
            VALUES (?, ?, ?, ?, ?)
        ''', data_list)
        self.conn.commit()
        print(f"★ DB保存完了: {len(data_list)}件のデータを保存しました。")

    def fetch_data_as_dataframe(self):
        return pd.read_sql("SELECT * FROM properties", self.conn)

    def close(self):
        self.conn.close()

# スクレイパークラス（デバッグ機能追加）
class SuumoScraper:
    def __init__(self, base_url):
        self.base_url = base_url
        self.headers = {
            # 一般的なブラウザに見せかけるためのUser-Agent
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }

    def get_html(self, url):
        try:
            time.sleep(3) # [cite: 36] サーバー負荷への配慮
            print(f"アクセス中: {url}")
            response = requests.get(url, headers=self.headers)
            response.encoding = response.apparent_encoding # 文字化け対策
            
            print(f"ステータスコード: {response.status_code}")
            if response.status_code != 200:
                print("⚠️ アクセスに失敗しました。")
                return None
                
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Error fetching URL: {e}")
            return None

    def parse_html(self, html):
        soup = BeautifulSoup(html, 'html.parser')
        property_list = []
        
        # 物件ブロックを取得
        items = soup.find_all("div", class_="cassetteitem")
        print(f"ページ内で見つかった物件ブロック数: {len(items)}")

        # ★もし0件なら、HTMLの中身を保存して確認する
        if len(items) == 0:
            with open("debug.html", "w", encoding="utf-8") as f:
                f.write(html)
            print("⚠️ 物件が見つかりませんでした。'debug.html' を作成しましたので、ブラウザで開いて中身を確認してください（ロボット判定されている可能性があります）。")
            return []

        for item in items:
            try:
                # 基本情報の取得
                name = item.find("div", class_="cassetteitem_content-title").text.strip()
                address = item.find("li", class_="cassetteitem_detail-col1").text.strip()
                station = item.find("li", class_="cassetteitem_detail-col2").text.strip()
                
                age_text = item.find("li", class_="cassetteitem_detail-col3").find_all("div")[0].text.strip()
                if "新築" in age_text:
                    age = 0
                else:
                    age = int(re.sub(r"\D", "", age_text)) if re.search(r"\d", age_text) else 0

                # 部屋情報の取得（1つの物件に複数の部屋がある場合がある）
                tbody = item.find("table", class_="cassetteitem_other").find("tbody")
                for tr in tbody.find_all("tr"):
                    try:
                        # 家賃の取得（階数や管理費などの列ズレに注意）
                        # 4列目が家賃であることが一般的だが、構造が変わるとここで失敗する
                        price_tds = tr.find_all("td")
                        price_text = price_tds[3].find("span", class_="cassetteitem_other-emphasisui").text.strip()
                        
                        price = float(price_text.replace("万円", "")) * 10000
                        
                        property_data = (name, address, price, age, station)
                        property_list.append(property_data)
                    except Exception as e:
                        # 部屋ごとの取得エラー（詳細表示）
                        # print(f"部屋情報の取得スキップ: {e}") 
                        continue

            except Exception as e:
                print(f"物件情報の取得エラー: {e}")
                continue
                    
        return property_list

    def run(self, target_url, max_pages=1):
        all_data = []
        for page in range(1, max_pages + 1):
            url = f"{target_url}&page={page}"
            html = self.get_html(url)
            if html:
                data = self.parse_html(html)
                all_data.extend(data)
                print(f"現在確保済みデータ数: {len(all_data)}件")
        return all_data

# --- 実行部分 ---
if __name__ == "__main__":
    # ★重要★ ここに検索結果の正しいURLを入れてください
    # 例: 東京23区全体の検索結果URLなど
    target_search_url = "https://suumo.jp/jj/chintai/ichiran/FR301FC001/?ar=030&bs=040&ta=13&sc=13101&sc=13102&sc=13103&cb=0.0&ct=9999999&mb=0&mt=9999999&et=9999999&cn=9999999&shkr1=03&shkr2=03&shkr3=03&shkr4=03&fw2="
    
    scraper = SuumoScraper(target_search_url)
    scraped_data = scraper.run(target_search_url, max_pages=1) # まずは1ページだけ試す
    
    if len(scraped_data) > 0:
        db = RealEstateDB()
        db.insert_data(scraped_data)
        df = db.fetch_data_as_dataframe()
        print("\n--- 取得データ確認 ---")
        print(df.head())
        db.close()
    else:
        print("\n❌ データが1件も取得できませんでした。URLかサイト構造を確認してください。")

SyntaxError: invalid syntax (1069067947.py, line 53)